# Deep Q-Networks

**Companion lesson:** https://ml-viz.vercel.app/courses/reinforcement-learning/03-deep-q-networks

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## DQN machinery in numpy

When states are too many to tabulate, approximate $Q$ with a network. We build a 2-layer MLP Q-network (one-hot state in, 4 Q-values out) plus the two tricks that make DQN stable: an **experience replay** buffer and a **target network**.

In [ ]:
class GridWorld:
    """n x n grid. Start top-left (0), goal bottom-right. Actions: 0=up 1=down 2=left 3=right.
    Reward -1 per step, +10 at the goal (terminal)."""
    def __init__(self, n=5):
        self.n = n; self.nS = n*n; self.nA = 4; self.goal = n*n-1
    def step(self, s, a):
        r, c = divmod(s, self.n)
        if a==0: r = max(0, r-1)
        elif a==1: r = min(self.n-1, r+1)
        elif a==2: c = max(0, c-1)
        else: c = min(self.n-1, c+1)
        s2 = r*self.n + c
        done = (s2 == self.goal)
        return s2, (10.0 if done else -1.0), done

env = GridWorld(5)

def one_hot(s, n):
    v = np.zeros(n); v[s] = 1; return v

def relu(x): return np.maximum(0, x)

class QNet:
    def __init__(self, n_in, nh, n_out, seed=0):
        rng = np.random.RandomState(seed)
        self.W1 = rng.randn(n_in, nh)*np.sqrt(2/n_in); self.b1 = np.zeros(nh)
        self.W2 = rng.randn(nh, n_out)*np.sqrt(2/nh);  self.b2 = np.zeros(n_out)
    def forward(self, X):
        self.X = X; self.z1 = X@self.W1 + self.b1; self.h = relu(self.z1)
        return self.h @ self.W2 + self.b2
    def copy_from(self, other):
        self.W1, self.b1 = other.W1.copy(), other.b1.copy()
        self.W2, self.b2 = other.W2.copy(), other.b2.copy()
print('Q-network ready')

## Training loop: replay + target net

Each step is stored in the buffer; we train on random minibatches whose targets come from a **frozen** target network, synced every few hundred steps.

In [ ]:
from collections import deque
import random

def train_dqn(env, episodes=600, gamma=0.9, lr=0.01, nh=64, batch=32, sync=200):
    rng = np.random.RandomState(0); random.seed(0)
    q, qt = QNet(env.nS, nh, env.nA), QNet(env.nS, nh, env.nA)
    qt.copy_from(q)
    buf = deque(maxlen=5000); eps, step, returns = 1.0, 0, []
    for ep in range(episodes):
        s, done, total, t = 0, False, 0, 0
        while not done and t < 100:
            a = rng.randint(env.nA) if rng.rand()<eps else int(np.argmax(q.forward(one_hot(s,env.nS))))
            s2, r, done = env.step(s, a)
            buf.append((s,a,r,s2,done)); s=s2; total+=r; t+=1; step+=1
            if len(buf) >= batch:
                B = random.sample(buf, batch)
                S  = np.array([one_hot(b[0],env.nS) for b in B])
                S2 = np.array([one_hot(b[3],env.nS) for b in B])
                A = np.array([b[1] for b in B]); R = np.array([b[2] for b in B])
                D = np.array([b[4] for b in B], float)
                target = R + gamma*qt.forward(S2).max(1)*(1-D)   # frozen target net
                pred = q.forward(S)                              # caches activations
                dout = np.zeros_like(pred)
                dout[np.arange(batch), A] = (pred[np.arange(batch),A] - target)/batch
                # backprop through the 2-layer MLP
                dW2 = q.h.T@dout; db2 = dout.sum(0)
                dh = dout@q.W2.T; dz1 = dh*(q.z1>0)
                dW1 = S.T@dz1; db1 = dz1.sum(0)
                q.W2-=lr*dW2; q.b2-=lr*db2; q.W1-=lr*dW1; q.b1-=lr*db1
            if step % sync == 0: qt.copy_from(q)
        eps = max(0.05, eps*0.99); returns.append(total)
    return q, returns

q, returns = train_dqn(env)
print('trained DQN over', len(returns), 'episodes')

## Did it learn? Greedy rollout

In [ ]:
ma = np.convolve(returns, np.ones(30)/30, mode='valid')
plt.plot(ma, color='#6366f1'); plt.xlabel('episode'); plt.ylabel('return (30-ep avg)')
plt.title('DQN learning curve on the gridworld'); plt.show()

s, path, done = 0, [0], False
while not done and len(path) < 25:
    s, _, done = env.step(s, int(np.argmax(q.forward(one_hot(s,env.nS))))); path.append(s)
print('greedy path:', path)
print('reached goal:', path[-1]==env.goal, 'in', len(path)-1, 'steps')

## Key takeaways

- A **Q-network** replaces the table so values generalize across states.
- **Experience replay** decorrelates the data fed to gradient descent.
- A **target network** keeps the regression target from chasing its own tail.
- Remove either trick and the same loop becomes unstable.